# LAPORAN PRAKTIKUM KECERDASAN ARTIFISIAL LANJUT
**Nama: Sayyidah Fatimah Azzahra**

**NIM: 235150200111064**

**Kelas: KAL-E**

---
## Bab 5 : Klasifikasi Decision Tree dan XGBoost

### Decision Tree

#### 1) Import Data

Praktikum kali ini menggunakan dataset [Car Evaluation Dataset](https://archive.ics.uci.edu/ml/datasets/Car+Evaluation) dari UCI Machine Learning Repository. Dataset ini telah digunakan pada praktikum sebelumnya. Detail fitur dapat Anda pelajari pada link yang tersedia.

Unduh dataset yang akan digunakan pada praktikum kali ini. Anda dapat menggunakan aplikasi wget untuk mendowload dataset dan menyimpannya dalam Google Colab. Jalankan cell di bawah ini untuk mengunduh dataset

In [249]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [250]:
url = "https://github.com/adikara-ub/praktikum-ai-lanjut/raw/main/dataset/car_sample.csv"

Setelah dataset berhasil diunduh, langkah berikutnya adalah membaca dataset dengan memanfaatkan fungsi **readcsv** dari library pandas. Lakukan pembacaan berkas csv ke dalam dataframe dengan nama **data** menggunakan fungsi **readcsv**. Jangan lupa untuk melakukan import library pandas terlebih dahulu


In [251]:
data = pd.read_csv(url)



Cek isi dataset Anda dengan menggunakan perintah **head()**

In [252]:
data.head()

,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


#### 2) Membagi data menjadi data latih dan data uji

Metode pembelajaran mesin memerlukan dua jenis data :


1.   Data latih : Digunakan untuk proses training metode klasifikasi
2.   Data uji : Digunakan untuk proses evaluasi metode klasifikasi

Data uji dan data latih perlu dibuat terpisah (mutualy exclusive) agar hasil evaluasi lebih akurat.

Data uji dan data latih dapat dibuat dengan cara membagi dataset dengan rasio tertentu, misalnya 80% data latih dan 20% data uji.

Library Scikit-learn memiliki fungsi [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) pada modul **model_selection** untuk membagi dataset menjadi data latih dan data uji. Bagilah dataset anda menjadi dua, yaitu **data_latih** dan **data_uji**. Agar pengacakan data dilakukan secara konstan, parameter **random_state** diisi dengan nilai integer tertentu, pada praktikum ini diset 101. Kemudian, nilai indeks pada data latih dan data uji diatur ulang agar berurutan nilainya


In [ ]:
from sklearn.model_selection import train_test_split
data_latih, data_uji = train_test_split(data, test_size=0.2, random_state=101)
data_latih.reset_index(drop=True)
data_uji.reset_index(drop=True)
data_latih = data_latih.reset_index(drop=True)
data_uji = data_uji.reset_index(drop=True)


,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,high,4,more,med,med,unacc
1,low,med,2,more,small,high,unacc
2,vhigh,low,5more,2,big,med,unacc
3,low,vhigh,3,more,small,med,unacc
4,med,low,3,more,small,low,unacc
...,...,...,...,...,...,...,...
341,high,med,2,more,med,med,unacc
342,low,med,4,more,big,med,good
343,vhigh,vhigh,3,2,big,med,unacc
344,vhigh,low,2,4,med,low,unacc


Tampilkan banyaknya data pada **data_latih** dan **data_uji**. Seharusnya **data_latih** terdiri dari 208 data, dan **data_uji** terdiri dari 52 data

In [254]:
print(data_uji.shape[0])
print(data_latih.shape[0])

346
1382


#### 3) Menghitung Gini

Nilai Gini merupakan salah satu kriteria penentu variabel apa yang akan digunakan untuk membentuk cabang pada decision tree. Variabel dengan nilai Gini terbesar akan digunakan sebagai pembentukan cabang

Buatlah fungsi bernama **hitung_gini** yang berfungsi menghitung nilai Gini dari suatu nilai pada sebuah variabel

In [255]:
def hitung_gini (kolom_kelas):
    elemen,banyak = np. unique(kolom_kelas, return_counts = True)
    nilai_gini = 1 - np.sum([(banyak[i]/np.sum(banyak))**2 for i in range(len(elemen))])
    return nilai_gini

Buatlah fungsi bernama **gini_split** yang digunakan untuk menghitung nilai Gini keseluruhan dari sebuah variabel.

In [256]:
def gini_split(data, nama_fitur_split, nama_fitur_kelas):
    nilai,banyak= np.unique(data[nama_fitur_split],return_counts=True)
    gini_split = np.sum([(banyak[i] / np.sum(banyak)) * hitung_gini(data.where(data[nama_fitur_split]==nilai[i]).dropna() [nama_fitur_kelas]) for i in range(len(nilai))])
    return gini_split

Ujilah fungsi **gini_split** menggunakan data_latih pada variabel **buying** dan variabel kelas bernama **class**.

In [257]:
gini_split(data_latih, "buying", "class")

np.float64(0.4498424838345615)

#### 4) Pembentukan pohon

Pembentukan pohon dilakukan secara rekursif. Seperti metode rekursif pada umumnya, perlu ditentukan kondisi berhenti terlebih dahulu. Kondisi berhenti pada pembentukan pohon adalah:


1.   Jika hanya ada satu kelas pada data, kembalikan kelas tersebut
2.   Jika fitur data  = 0 (tidak ada fitur yang tersisa), kembalikan kelas dari parent
3. Jika data kosong (tidak ada data), kembalikan kelas dengan frekuensi terbanyak

Selain kondisi berhenti tersebut, dilakukan pembentukan pohon secara rekursif menggunakan fungsi **buat_tree**.



In [258]:
def buat_tree(data, data_awal, daftar_fitur, nama_fitur_kelas, kelas_parent_node=None):
    # jika hanya ada satu kelas, return kelas
    if len(np.unique(data[nama_fitur_kelas])) <= 1:
        return np.unique (data[nama_fitur_kelas])[0]

    # jika data kosong
    elif len(data)==0:
        return np.unique(data_awal[nama_fitur_kelas])[
            np.argmax(np.unique(data_awal[nama_fitur_kelas],return_counts=True)[1])]
    
    # jika tidak ada fitur yang tersisa
    elif len(daftar_fitur) ==0:
        return kelas_parent_node
    else:
        kelas_parent_node = np. unique(data[nama_fitur_kelas])
        [np.argmax(np.unique(data[nama_fitur_kelas],return_counts=True)[1])]
        nilai_split = [gini_split(data, fitur,nama_fitur_kelas) for fitur in daftar_fitur]
    
        index_fitur_terbaik = np.argmin(nilai_split)
        fitur_terbaik = daftar_fitur [index_fitur_terbaik]
        tree = {fitur_terbaik:{}}
        daftar_fitur = [i for i in daftar_fitur if i != fitur_terbaik]
        
        for nilai in np.unique(data[fitur_terbaik]):
            sub_data = data.where (data[fitur_terbaik] == nilai).dropna()
            subtree = buat_tree(sub_data, data_awal, daftar_fitur, nama_fitur_kelas, kelas_parent_node)
            tree[fitur_terbaik][nilai]=subtree
    return (tree)

Buatlah tree menggunakan data latih yang tersedia

In [259]:
tree = buat_tree(data_latih, data_latih,data_latih.columns[:-1], 'class')

Tampilkan tree yang terbentuk. Gunakan library **pprint** untuk menampilkan dictionary secara teratur.

In [260]:
from pprint import pprint

pprint(tree)

{'safety': {'high': {'persons': {'2': 'unacc',
                                 '4': {'buying': {'high': {'maint': {'high': 'acc',
                                                                     'low': 'acc',
                                                                     'med': 'acc',
                                                                     'vhigh': 'unacc'}},
                                                  'low': {'maint': {'high': {'lug_boot': {'big': 'vgood',
                                                                                          'med': {'doors': {'2': 'acc',
                                                                                                            '3': 'acc',
                                                                                                            '4': 'vgood'}},
                                                                                          'small': 'acc'}},
                                    

#### 5) Proses Prediksi

Proses prediksi kelas pada data uji dilakukan dengan melakukan *tree traversal* sampai menemui leaf.

In [261]:
def prediksi(data_uji,tree):
    for key in list(data_uji.keys()):
        if key in list(tree.keys()):
            try:
                hasil = tree[key][data_uji[key]]
            except:
                return 1
            hasil = tree[key][data_uji[key]]
            if isinstance(hasil,dict):
                return prediksi(data_uji,hasil)
            else:
                return hasil

#### 6) Proses Pengujian
Lakukan pengujian menggunakan data uji. Kelas pada data uji perlu dihapus dan data uji perlu diubah menjadi dictionary

In [262]:
data_uji_dict = data_uji. iloc[:,:-1].to_dict(orient = "records")

Lakukan pengujian terhadap keseluruhan data uji menggunakan looping.

In [263]:
hasil_prediksi_total = []
for i in range(len(data_uji_dict)):
    hasil_prediksi = prediksi(data_uji_dict[i],tree)
    hasil_prediksi_total.append(hasil_prediksi)

Bandingkan hasil prediksi dengan label sebenarnya. Hitunglah banyaknya data uji yang memiliki kelas prediksi sama dengan kelas sebenarnya

In [264]:
print("Total prediksi benar:",sum(hasil_prediksi_total==data_uji['class']))
print("Total prediksi salah:",sum(hasil_prediksi_total!=data_uji['class']))
print("Akurasi: ", sum(hasil_prediksi_total==data_uji['class'])/len(data_uji))
print("Total data: ", len(data_uji))

Total prediksi benar: 298
Total prediksi salah: 48
Akurasi:  0.861271676300578
Total data:  346


## XGBoost

#### 1) Instalasi module/package/library XGBoost

In [265]:
!pip install xgboost

#### 2) Impor Data

In [266]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

#### 3) Pembuatan Classifier dan Model

In [267]:
df = data.copy()

In [268]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df_new = df.apply(label_encoder.fit_transform)

X_train, X_test, y_train, y_test = train_test_split(df_new.loc[:,'buying': 'safety'],
df_new['class'], test_size=.2)

bst = XGBClassifier(n_estimators=2, max_depth=2, learning_rate=1,objective='binary:logistic')

#### 4) Proses Pengujian/Prediksi

In [269]:
bst.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=2,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=2,
              n_jobs=None, num_parallel_tree=None, ...)

In [270]:
y_pred = bst.predict(X_test)

In [271]:
print(y_pred)

[2 2 0 2 0 2 2 0 0 2 2 2 0 2 2 0 2 2 0 2 2 0 0 2 2 2 2 0 0 0 2 2 2 0 0 2 2
 0 2 2 2 0 2 2 0 0 0 0 2 2 0 2 0 2 0 0 2 0 2 0 0 0 0 2 0 0 2 2 2 0 0 2 2 0
 0 0 2 0 0 2 2 2 0 2 2 2 0 0 2 2 2 2 2 2 2 0 0 2 0 0 0 2 0 2 0 0 2 2 2 2 0
 2 2 2 2 0 0 0 0 2 2 0 0 0 0 0 0 2 0 2 2 2 2 2 2 2 0 0 0 0 0 0 0 0 2 0 0 0
 2 0 0 0 0 2 2 2 2 2 0 0 0 0 0 2 0 2 0 0 2 0 0 2 2 2 0 2 0 2 2 2 2 0 0 2 2
 0 0 2 2 0 0 0 0 0 2 2 2 2 0 2 2 0 0 0 0 0 0 2 0 0 0 2 0 0 0 0 2 0 0 2 0 2
 2 2 0 2 2 0 2 0 0 0 0 0 0 2 0 0 2 2 0 0 2 2 2 2 0 0 2 2 2 0 2 2 2 0 2 0 2
 0 0 2 2 2 2 0 0 2 2 2 2 2 0 0 0 2 0 0 2 2 2 0 2 2 0 0 0 2 2 0 2 2 2 2 0 2
 0 0 2 2 2 2 2 0 2 0 2 2 2 2 2 2 0 0 2 0 2 2 2 2 2 2 0 2 0 0 2 0 2 0 0 2 0
 2 2 2 2 0 0 2 2 2 0 0 0 0]


In [272]:
y_label = label_encoder.inverse_transform(y_pred)
print(y_label)

['unacc' 'unacc' 'acc' 'unacc' 'acc' 'unacc' 'unacc' 'acc' 'acc' 'unacc'
 'unacc' 'unacc' 'acc' 'unacc' 'unacc' 'acc' 'unacc' 'unacc' 'acc' 'unacc'
 'unacc' 'acc' 'acc' 'unacc' 'unacc' 'unacc' 'unacc' 'acc' 'acc' 'acc'
 'unacc' 'unacc' 'unacc' 'acc' 'acc' 'unacc' 'unacc' 'acc' 'unacc' 'unacc'
 'unacc' 'acc' 'unacc' 'unacc' 'acc' 'acc' 'acc' 'acc' 'unacc' 'unacc'
 'acc' 'unacc' 'acc' 'unacc' 'acc' 'acc' 'unacc' 'acc' 'unacc' 'acc' 'acc'
 'acc' 'acc' 'unacc' 'acc' 'acc' 'unacc' 'unacc' 'unacc' 'acc' 'acc'
 'unacc' 'unacc' 'acc' 'acc' 'acc' 'unacc' 'acc' 'acc' 'unacc' 'unacc'
 'unacc' 'acc' 'unacc' 'unacc' 'unacc' 'acc' 'acc' 'unacc' 'unacc' 'unacc'
 'unacc' 'unacc' 'unacc' 'unacc' 'acc' 'acc' 'unacc' 'acc' 'acc' 'acc'
 'unacc' 'acc' 'unacc' 'acc' 'acc' 'unacc' 'unacc' 'unacc' 'unacc' 'acc'
 'unacc' 'unacc' 'unacc' 'unacc' 'acc' 'acc' 'acc' 'acc' 'unacc' 'unacc'
 'acc' 'acc' 'acc' 'acc' 'acc' 'acc' 'unacc' 'acc' 'unacc' 'unacc' 'unacc'
 'unacc' 'unacc' 'unacc' 'unacc' 'acc' 'acc' 'acc' 'a

## TUGAS
Pada tugas kali ini Anda diminta memodifikasi metode pembentukan tree yang telah Anda agar metode tersebut menggunakan information gain sebagai dasar percabangan. Lengkapilah kerangka source code di bawah ini

Lengkapi fungsi hitung_entropy

In [273]:
import numpy as np
from scipy.stats import entropy
from math import log, e
import pandas as pd
import math

In [274]:
'''
entropy(p) = − SUM (Pi * log(Pi) )
'''

def hitung_entropy(kolom_kelas):
  class_counts = kolom_kelas.value_counts()
  total_samples = len(kolom_kelas)
  entropy = 0
  for count in class_counts:
    p_i = count / total_samples
    entropy -= p_i * math.log2(p_i)
  return entropy

Lengkapi fungsi information_gain

In [275]:
'''
gain(D, A) = entropy(D)−􏰋 SUM ( |Di| / |D| * entropy(Di) )
'''

def information_gain(data, nama_fitur_split, nama_fitur_kelas):
  total_entropy = hitung_entropy(data[nama_fitur_kelas])
  values = data[nama_fitur_split].unique()
  weighted_entropy = 0
  total_samples = len(data)

  for value in values:
    subset = data[data[nama_fitur_split] == value]
    subset_entropy = hitung_entropy(subset[nama_fitur_kelas])
    weighted_entropy += (len(subset) / total_samples) * subset_entropy

  information_gain = total_entropy - weighted_entropy

  return information_gain

Lengkapi fungsi **buat_tree_ig**. Isinya sama persis dengan fungsi **buat_tree**, hanya saja penghitungan **gini_split** diganti dengan **information_gain**. Selain itu, percabangan dilakukan dengan menggunakan nilai **information_gain** **terbesar**

In [276]:
def buat_tree_ig(data, data_awal, daftar_fitur, nama_fitur_kelas, kelas_parent_node=None):
    # kalau data kosong, return parent node
    if len(data) == 0:
        return kelas_parent_node
    
    # kalau seluruh sampel memiliki kelas yang sama, return kelas tersebut
    if len(data[nama_fitur_kelas].unique()) == 1:
        return data[nama_fitur_kelas].iloc[0]
    
    # kalau tidak ada fitur yang bisa dipakai, return kelas paling banyak
    if len(daftar_fitur) == 0:
        return data[nama_fitur_kelas].value_counts().idxmax()
    
    kelas_parent_node = data[nama_fitur_kelas].value_counts().idxmax()
    
    # mencari fitur terbaik untuk dijadikan root (fitur dengan IG terbesar)
    nilai_ig_terbaik = -1
    fitur_terbaik = None
    
    for fitur in daftar_fitur:
        nilai_ig = information_gain(data, fitur, nama_fitur_kelas)
        if nilai_ig > nilai_ig_terbaik:
            nilai_ig_terbaik = nilai_ig
            fitur_terbaik = fitur
    
    # kalau tidak ada information gain, return kelas paling banyak
    if nilai_ig_terbaik <= 0:
        return kelas_parent_node
    
    # buat tree dengan fitur terbaik sebagai root
    tree = {fitur_terbaik: {}}
    
    nilai_fitur = data_awal[fitur_terbaik].unique()
    
    # hapus fitur terbaik dari daftar fitur
    daftar_fitur_baru = [f for f in daftar_fitur if f != fitur_terbaik]
    
    # membuat subtree untuk setiap nilai fitur terbaik
    for nilai in nilai_fitur:
        subset = data[data[fitur_terbaik] == nilai]
        subtree = buat_tree_ig(subset, data_awal, daftar_fitur_baru, nama_fitur_kelas, kelas_parent_node)
        tree[fitur_terbaik][nilai] = subtree
    
    return tree

Lakukan pembentukan tree menggunakan fungsi **buat_tree_ig**

In [277]:
tree_ig = buat_tree_ig(data_latih,data_latih,data_latih.columns[:-1],'class')

Tampilkan tree yang terbentuk

In [278]:
pprint(tree_ig)

{'safety': {'high': {'persons': {'2': 'unacc',
                                 '4': {'buying': {'high': {'maint': {'high': 'acc',
                                                                     'low': 'acc',
                                                                     'med': 'acc',
                                                                     'vhigh': 'unacc'}},
                                                  'low': {'maint': {'high': {'lug_boot': {'big': 'vgood',
                                                                                          'med': {'doors': {'2': 'acc',
                                                                                                            '3': 'acc',
                                                                                                            '4': 'vgood',
                                                                                                            '5more': 'acc'}},
                    

Lakukan pengujian menggunakan tree yang terbentuk

In [279]:
hasil_prediksi_total_ig = []
for i in range(len(data_uji_dict)):
  hasil_prediksi = prediksi(data_uji_dict[i],tree_ig)
  hasil_prediksi_total_ig.append(hasil_prediksi)
print("Total prediksi benar: ",sum(hasil_prediksi_total_ig==data_uji['class']))
print("Total prediksi salah: ",len(data_uji_dict)-sum(hasil_prediksi_total_ig==data_uji['class']))
print("Akurasi: ",sum(hasil_prediksi_total_ig==data_uji['class'])/len(data_uji_dict))
print("Total data: ",len(data_uji_dict))

Total prediksi benar:  310
Total prediksi salah:  36
Akurasi:  0.8959537572254336
Total data:  346


### PERTANYAAN

Jawablah pertanyaan di bawah ini



1.   Amati tree yang dihasilkan dengan kriteria percabangan GINI dan Information Gain. Apa perbedaan tree yang dihasilkan dari kedua metode tersebut?
2.   Apakah penggunaan Information Gain dapat meningkatkan akurasi prediksi?



### nomor satu

Berdasarkan struktur kedua tree, perbedaan yang dimiliki kedua tree ada pada struktur meski keduanya sama-sama menggunakan atribut 'safety' sebagai root node.
Contoh:
1. Tree Gini Index punya struktur cabang yang sama pada beberapa bagian, misalnya pada cabang 'safety' → 'high' → 'persons' → '4' → 'buying' → 'low', sedangkan Tree Information Gain menunjukkan perbedaan struktur pada nilai doorsnya, seperti pada cabang 'med' → 'doors' dimana ada perbedaan nilai '5more' antara kedua tree.
2. Tree Information Gain cenderung membuat keputusan klasifikasi lebih awal pada beberapa jalur (cabang jadi lebih pendek), sementara tree Gini Index cenderung membuat keputusan klasifikasi lebih lambat pada beberapa jalur (cabang jadi lebih panjang).
3. Ada beberapa jalur decision yang menghasilkan prediksi berbeda antara kedua tree, misalnya pada jalur 'safety' → 'med' → 'persons' → '4' → 'buying' → 'high' → 'lug_boot' → 'med' → 'doors' → '5more' dimana pada tree Gini menghasilkan 'unacc' sedangkan pada tree Information Gain menghasilkan 'acc'.


### nomor dua

Berikut informasi hasil prediksi dari setiap model:<br>

**Tree GINI:**<br>
Total prediksi benar: 298<br>
Total prediksi salah: 48<br>
Total data:  346<br>
Akurasi:  0.8612 (86%)<br>

**Tree Information Gain:**<br>
Total prediksi benar: 310<br>
Total prediksi salah: 36<br>
Total data: 346 <br>
Akurasi: 0.8960 (89.60%)<br>


Sehingga, dapat disimpulkan bahwa penggunaan Information Gain lebih efektif dalam mengidentifikasi fitur yang memberikan pemisahan kelas optimal untuk kasus ini.